<a href="https://colab.research.google.com/github/user903taken/Bank-management-system/blob/main/bear__classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install voila
!jupyter serverextension enable --sys-prefix voila

Enabling: voila
- Writing config: /usr/etc/jupyter
    - Validating...
      voila 0.5.13 OK


In [ ]:
!pip install -Uqq fastbook
import fastbook
from fastbook import *
from fastai.vision.widgets import *
fastbook.setup_book()

In [ ]:
from google.colab import userdata
key = userdata.get('SERP_API_KEY')  # add this in Colab's Secrets tab, enable notebook access

In [ ]:
bear_types = 'grizzly', 'black', 'teddy'
path = Path('bears')

if not path.exists():
    path.mkdir()
    for o in bear_types:
        dest = path/o
        dest.mkdir(exist_ok=True)
        results = search_images_serpapi(f'{o} bear', key=key)
        download_images(dest, urls=results)

fns = get_image_files(path)
failed = verify_images(fns)
failed.map(Path.unlink);
len(fns), len(failed)

(307, 0)

In [ ]:
bears = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=RandomResizedCrop(224, min_scale=0.5),
    batch_tfms=aug_transforms())
dls = bears.dataloaders(path)

learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(4)

epoch,train_loss,valid_loss,error_rate,time
0,1.783746,0.143344,0.016393,01:34


epoch,train_loss,valid_loss,error_rate,time
0,0.304087,0.040700,0.000000,01:49
1,0.226297,0.015589,0.000000,01:25
2,0.153937,0.008396,0.000000,01:24
3,0.115903,0.006999,0.000000,01:24


In [ ]:
learn.export()
learn_inf = load_learner(Path()/'export.pkl')
learn_inf.dls.vocab

btn_upload = widgets.FileUpload()
btn_upload

/usr/local/lib/python3.13/dist-packages/fastai/learner.py:456: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


FileUpload(value={}, description='Upload')

In [ ]:
out_pl = widgets.Output()
lbl_pred = widgets.Label()
btn_run = widgets.Button(description='Classify')

def on_click_classify(change):
    img = PILImage.create(btn_upload.data[-1])
    out_pl.clear_output()
    with out_pl: display(img.to_thumb(128, 128))
    pred, pred_idx, probs = learn_inf.predict(img)
    lbl_pred.value = f'Prediction: {pred}; Probability: {probs[pred_idx]:.04f}'

btn_run.on_click(on_click_classify)
VBox([widgets.Label('Select your bear!'), btn_upload, btn_run, out_pl, lbl_pred])